# AgentCore Gateway에서 프라이빗 도메인 사용하기(Private DNS)

이 실습에서는 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)를 **프라이빗 도메인**, 즉 [Route 53 프라이빗 호스팅 영역](https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/hosted-zones-private.html)을 통해 VPC 내부에서만 확인되는 도메인을 사용하는 리소스에 연결하는 방법을 알아봅니다.

VPC에서 **Private DNS**가 활성화되어 있으면(기본값), AgentCore Gateway의 관리형 Resource Gateway가 VPC의 DNS 해석기를 통해 프라이빗 도메인을 확인합니다.

## 아키텍처

![아키텍처](./images/private-domain.png)

- 대상 URL은 프라이빗 FQDN을 사용합니다(예: `https://internal.yourcompany.com`).
- VPC에 연결된 해당 FQDN용 Route 53 **프라이빗 호스팅 영역**이 도메인의 별칭을 내부 ALB로 지정합니다.
- ALB는 동일한 FQDN에 대해 공개적으로 신뢰할 수 있는 ACM 인증서를 사용하여 TLS를 종료합니다.
- AgentCore Gateway의 Resource Gateway가 Private DNS를 통해 도메인을 확인합니다 → ALB의 프라이빗 IP를 가져옵니다 → 퍼블릭 인증서로 TLS 연결을 설정합니다 → 요청이 백엔드에 도달합니다.

VPC 송신 및 관리형 VPC 리소스에 관한 배경 정보는 [프로젝트 README](../README.md)와 [고급 개념 README](./README.md)를 참조하세요.

## Private DNS의 작동 방식

VPC Lattice의 관리형 Resource Gateway(AgentCore가 `CreateGatewayTarget`을 `managedVpcResource`와 함께 호출할 때 생성)는 VPC의 DNS 해석기를 사용하여 대상 엔드포인트 도메인을 조회합니다. VPC가 해당 도메인의 Route 53 프라이빗 호스팅 영역에 연결되어 있으면 해석기가 그 영역의 레코드를 반환합니다.

### 요구 사항

- **VPC DNS 활성화** — VPC에서 `enableDnsSupport`와 `enableDnsHostnames`가 모두 `true`여야 합니다(기본값이며 워크숍의 `VpcegressStack`에서 설정됨).
- **프라이빗 호스팅 영역 연결** — 호스팅 영역은 Resource Gateway ENI가 있는 VPC에 연결되어야 합니다.
- **공개적으로 신뢰할 수 있는 TLS 인증서** — ALB는 대상 FQDN을 포함하는 퍼블릭 CA 발급 인증서(ACM 퍼블릭 인증서)를 제시해야 합니다. AgentCore Gateway는 퍼블릭 루트 CA를 기준으로 인증서를 검증합니다.
- **인증서에 포함된 대상 FQDN** — `endpoint` URL에 전달하는 도메인은 인증서의 주체/SAN과 일치해야 합니다.

> **HTTPS를 사용하지 않는 백엔드인가요?** 백엔드가 TLS를 사용하지 않는 경우(예: 포트 8000의 일반 HTTP 서비스), 이 실습의 ALB가 TLS 종료를 처리합니다. 백엔드가 소유한 도메인의 퍼블릭 인증서로 이미 TLS를 종료한다면 ALB를 생략하고 프라이빗 호스팅 영역이 백엔드를 직접 가리키도록 할 수 있습니다. 백엔드가 **프라이빗** 인증서를 사용하는 경우 ALB + 호스트 헤더 변환 패턴은 [프라이빗 인증 기관](./02-private-certificate-authority.ipynb) 및 [자체 서명 인증서](./03-self-signed-certificate.ipynb) 실습을 참조하세요.

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포)
- 소유한 도메인의 [ACM 퍼블릭 인증서](../00-prerequisites/create-acm-public-certificate.md)

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

## 2단계: 도메인 및 인증서 구성

ACM 퍼블릭 인증서 ARN과 인증서가 포함하는 **프라이빗 FQDN**을 입력합니다. FQDN은 인증서의 주체/SAN과 일치해야 합니다. 예를 들어 인증서가 `internal.yourcompany.com`용이라면 여기에 `internal.yourcompany.com`을 입력합니다.

CDK 스택은 이 이름과 정확히 일치하는 **Route 53 프라이빗 호스팅 영역**을 생성하여 VPC에 연결하고, 내부 ALB를 가리키는 영역 정점 별칭 레코드를 추가합니다. VPC 내부에서 `https://<DOMAIN>`은 Private DNS를 통해 ALB의 프라이빗 IP로 확인되므로 퍼블릭 DNS 레코드가 필요하지 않습니다.

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ").strip()
DOMAIN = input("Domain name covered by the certificate (e.g., api.internal.yourcompany.com): ").strip()

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"
assert " " not in DOMAIN, "Domain must not contain whitespace"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## 3단계: 인프라 배포

이 스택은 다음 리소스를 배포합니다.
- HTTP 포트 8000에서 REST API(FastAPI)를 실행하는 **EC2 인스턴스**
- HTTPS 포트 443에서 퍼블릭 ACM 인증서를 사용하고 HTTP를 통해 EC2로 전달하는 **내부 ALB**
- VPC에 연결되고 ALB를 가리키는 **영역 정점 별칭** 레코드가 있는 `<DOMAIN>`이라는 이름의 **Route 53 프라이빗 호스팅 영역**

VPC 내부에서 `<DOMAIN>`은 ALB의 프라이빗 IP로 확인됩니다. VPC 외부에서는 도메인이 확인되지 않지만, AgentCore Gateway의 Resource Gateway는 VPC 내부에 있고 Private DNS를 사용하여 도메인을 조회하므로 문제가 되지 않습니다.

In [ ]:
!cdk deploy PrivateDomain \
    -c "publicCertArn={CERT_ARN}" \
    -c "privateDomain={DOMAIN}" \
    --profile {ACCOUNT_A_PROFILE} \
    --require-approval never \
    --outputs-file pd-outputs.json

In [ ]:
with open("pd-outputs.json") as f:
    pd_outputs = json.load(f)["PrivateDomain"]

ALB_DNS = pd_outputs["AlbDnsName"]
ALB_SG_ID = pd_outputs["AlbSgId"]
API_KEY_VALUE = pd_outputs["ApiKey"]
EC2_INSTANCE_ID = pd_outputs["Ec2InstanceId"]
EC2_PRIVATE_IP = pd_outputs["Ec2PrivateIp"]
PRIVATE_DOMAIN = pd_outputs["PrivateDomain"]

print(f"Private FQDN:     {PRIVATE_DOMAIN}  (resolves via Private DNS inside VPC → ALB)")
print(f"ALB DNS:          {ALB_DNS}")
print(f"ALB SG:           {ALB_SG_ID}")
print(f"EC2 instance:     {EC2_INSTANCE_ID}")
print(f"EC2 private IP:   {EC2_PRIVATE_IP}")

## 4단계: AgentCore Gateway 대상 생성

대상이 프라이빗 엔드포인트를 직접 가리키도록 합니다.

- **대상 URL**: `https://{DOMAIN}` — 프라이빗 호스팅 영역을 통해 ALB의 프라이빗 IP로 확인됩니다.
- **`managedVpcResource`**: Resource Gateway ENI가 포트 443에서 ALB에 연결할 수 있도록 VPC ID, 서브넷 ID 및 ALB의 보안 그룹을 지정합니다.

> **보안 그룹:** Resource Gateway ENI가 포트 443에서 ALB에 연결할 수 있도록 ALB의 보안 그룹을 `securityGroupIds`에 전달합니다.

In [ ]:
with open("03-advanced-concepts/openapi-private.json") as f:
    openapi_schema = json.load(f)

TARGET_ENDPOINT = f"https://{DOMAIN}"
openapi_schema["servers"] = [{"url": TARGET_ENDPOINT}]

OPENAPI_SCHEMA = json.dumps(openapi_schema)
print(f"Server URL:     {TARGET_ENDPOINT}  (resolves via Private DNS inside the VPC)")
print(f"Endpoints:      {list(openapi_schema['paths'].keys())}")

In [ ]:
# API 키 자격 증명 공급자 생성
cred_response = agentcore.create_api_key_credential_provider(
    name="private-domain-api-key",
    apiKey=API_KEY_VALUE,
)
CRED_PROVIDER_ARN = cred_response["credentialProviderArn"]
print(f"Credential provider ARN: {CRED_PROVIDER_ARN}")

In [ ]:
response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="private-domain",
    description="Private domain via Private DNS",
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "inlinePayload": OPENAPI_SCHEMA,
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "API_KEY",
            "credentialProvider": {
                "apiKeyCredentialProvider": {
                    "providerArn": CRED_PROVIDER_ARN,
                    "credentialParameterName": "x-api-key",
                    "credentialLocation": "HEADER",
                }
            },
        }
    ],
    privateEndpoint={
        "managedVpcResource": {
            "vpcIdentifier": VPC_USW2_ID,
            "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
            "endpointIpAddressType": "IPV4",
            "securityGroupIds": [ALB_SG_ID],
        }
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpointManagedResources', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 5단계: AgentCore Gateway를 통해 API 호출

Cognito에서 액세스 토큰을 가져온 다음 게이트웨이를 통해 API를 호출합니다.
대상의 프라이빗 도메인은 VPC의 DNS 해석기를 통해서만 확인되므로 외부에 공개되는 항목이 없습니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "private-domain___healthCheck", "arguments": {}},
        "id": 2,
    },
)
print("Health check:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# 항목 생성
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "private-domain___createItem",
            "arguments": {"name": "Widget", "price": 9.99},
        },
        "id": 3,
    },
)
print("Create item:")
print(json.dumps(response.json(), indent=2))

In [ ]:
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "private-domain___listItems", "arguments": {}},
        "id": 4,
    },
)
print("Items:")
print(json.dumps(response.json(), indent=2))

## 정리

1. 게이트웨이 대상 및 자격 증명 공급자 삭제
2. CDK 스택 제거
3. 유지된 ALB 보안 그룹 삭제

> **참고:** AgentCore의 관리형 Resource Gateway ENI가 ALB 보안 그룹을 계속 참조할 수 있으므로 스택을 삭제할 때 보안 그룹이 유지됩니다. 게이트웨이 대상이 완전히 제거된 후 수동으로 삭제하세요.

In [ ]:
# # 1단계: 게이트웨이 대상 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 자격 증명 공급자 삭제
# agentcore.delete_api_key_credential_provider(name="private-domain-api-key")
# print("Deleted credential provider: private-domain-api-key")

In [ ]:
# # 2단계: 스택 제거
# !cdk destroy PrivateDomain \
#     -c "publicCertArn={CERT_ARN}" \
#     -c "privateDomain={DOMAIN}" \
#     --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # 3단계: 유지된 ALB 보안 그룹 삭제
# # "DependencyViolation" 오류가 발생하면 ENI가 해제될 때까지 몇 분 정도 기다립니다.
# ec2_client = session.client("ec2")
# try:
#     ec2_client.delete_security_group(GroupId=ALB_SG_ID)
#     print(f"Deleted security group: {ALB_SG_ID}")
# except ec2_client.exceptions.ClientError as e:
#     if "DependencyViolation" in str(e):
#         print(f"SG {ALB_SG_ID} still has dependencies. Wait a few minutes and retry.")
#     else:
#         raise